# Project 2: NLP — Text Classification with TF-IDF + Multiple Classifiers

From-scratch pipeline: text cleaning, TF-IDF (unigram+bigram), SVM/LR/SGD/NB comparison, feature analysis.

In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Synthetic multi-class dataset (realistic text classification)
categories = {
    "sports"    : ["team scored goal match player season tournament championship league winner",
                   "football basketball soccer tennis game player coach athlete stadium fans",
                   "world cup final score penalty kick referee offside players training"],
    "tech"      : ["algorithm software computer programming code function data structure",
                   "machine learning neural network artificial intelligence deep model",
                   "python javascript cloud server database api framework development"],
    "finance"   : ["stock market investment portfolio dividend earnings revenue profit",
                   "bank interest rate inflation economy gdp recession growth fiscal",
                   "crypto bitcoin trading hedge fund equity bond yield currency"],
    "health"    : ["patient medical doctor hospital treatment disease symptoms diagnosis",
                   "vaccine drug clinical trial health research therapy prevention cure",
                   "blood pressure heart lung brain surgery medicine prescription chronic"],
}

docs, labels = [], []
np.random.seed(42)
for i, (cat, templates) in enumerate(categories.items()):
    for _ in range(600):
        # Sample 2-4 templates and combine with noise words
        chosen = np.random.choice(templates, size=np.random.randint(2,4), replace=True)
        noise  = np.random.choice(["the","a","is","was","and","of","to","in","for","with",
                                    "on","at","by","from","this","that","it","as","be","are"],
                                   size=np.random.randint(5,12))
        words  = " ".join(chosen).split() + list(noise)
        np.random.shuffle(words)
        docs.append(" ".join(words))
        labels.append(cat)

label_names = sorted(set(labels))
label2id = {l:i for i,l in enumerate(label_names)}
y = np.array([label2id[l] for l in labels])

print("Dataset shape:", len(docs), "documents,", len(label_names), "classes")
print("Classes:", label_names)
for i,cat in enumerate(label_names):
    print("  {:10s}: {:4d} samples".format(cat, (y==i).sum()))


Dataset shape: 2400 documents, 4 classes
Classes: ['finance', 'health', 'sports', 'tech']
  finance   :  600 samples
  health    :  600 samples
  sports    :  600 samples
  tech      :  600 samples


In [2]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", " NUM ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

docs_clean = [clean_text(d) for d in docs]

X_train, X_test, y_train, y_test = train_test_split(
    docs_clean, y, test_size=0.2, stratify=y, random_state=42
)

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2),
                        sublinear_tf=True, min_df=2, max_df=0.95)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)

print("Train:", X_tr.shape, " Test:", X_te.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))
avg_len = np.mean([len(d.split()) for d in docs_clean])
print("Avg doc length: {:.1f} tokens".format(avg_len))


Train: (1920, 7108)  Test: (480, 7108)
Vocabulary size: 7108
Avg doc length: 30.0 tokens


In [3]:
models = {
    "LinearSVC"    : LinearSVC(C=1.0, max_iter=2000),
    "LogisticReg"  : LogisticRegression(C=5.0, max_iter=500, solver="lbfgs"),
    "SGD"          : SGDClassifier(loss="hinge", max_iter=100, random_state=42),
    "MultinomialNB": MultinomialNB(alpha=0.1),
}

print("Model Comparison:")
best_name, best_acc, best_preds = None, 0, None
for name, clf in models.items():
    clf.fit(X_tr, y_train)
    preds = clf.predict(X_te)
    acc   = accuracy_score(y_test, preds)
    if acc > best_acc:
        best_acc, best_name, best_preds = acc, name, preds
    print("  {:15s}  Accuracy={:.4f}".format(name, acc))

print("\n=== Best Model: {} (Accuracy={:.4f}) ===".format(best_name, best_acc))
print(classification_report(y_test, best_preds, target_names=label_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, best_preds))


Model Comparison:
  LinearSVC        Accuracy=1.0000


  LogisticReg      Accuracy=1.0000
  SGD              Accuracy=1.0000
  MultinomialNB    Accuracy=1.0000

=== Best Model: LinearSVC (Accuracy=1.0000) ===
              precision    recall  f1-score   support

     finance       1.00      1.00      1.00       120
      health       1.00      1.00      1.00       120
      sports       1.00      1.00      1.00       120
        tech       1.00      1.00      1.00       120

    accuracy                           1.00       480
   macro avg       1.00      1.00      1.00       480
weighted avg       1.00      1.00      1.00       480

Confusion Matrix:
[[120   0   0   0]
 [  0 120   0   0]
 [  0   0 120   0]
 [  0   0   0 120]]


In [4]:
# Top discriminative terms per class (LinearSVC coef)
svc = models["LinearSVC"]
feat_names = tfidf.get_feature_names_out()
print("Top 12 discriminative terms per class:")
for i, cat in enumerate(label_names):
    coef    = svc.coef_[i]
    top_idx = coef.argsort()[-12:][::-1]
    terms   = [feat_names[j] for j in top_idx]
    print("  {:10s}: {}".format(cat, ", ".join(terms)))

# 5-fold cross-validation
print("\n5-Fold CV (LinearSVC):")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(LinearSVC(C=1.0, max_iter=2000), X_tr, y_train, cv=cv, scoring="accuracy")
print("  Mean Acc={:.4f}  Std={:.4f}".format(scores.mean(), scores.std()))
print("  Per-fold:", ["  {:.4f}".format(s) for s in scores])


Top 12 discriminative terms per class:
  finance   : profit, stock, market, investment, portfolio, dividend, revenue, earnings, growth, gdp, bank, interest
  health    : hospital, medical, diagnosis, treatment, doctor, symptoms, disease, patient, heart, chronic, lung, brain
  sports    : player, players, final, offside, penalty, training, score, world, kick, referee, cup, coach
  tech      : database, development, framework, javascript, server, python, api, cloud, neural, intelligence, network, model

5-Fold CV (LinearSVC):


  Mean Acc=1.0000  Std=0.0000
  Per-fold: ['  1.0000', '  1.0000', '  1.0000', '  1.0000', '  1.0000']
